# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os

# Clone the GitHub repository to access the data file
repo_url = 'https://github.com/d1vyesh-27/flyrank-ai-internship-ml'
repo_name = repo_url.split('/')[-1]

if not os.path.exists(repo_name):
    !git clone {repo_url}
else:
    print(f"Repository '{repo_name}' already cloned.")

# Change the current working directory to the cloned repository
os.chdir(repo_name)

Cloning into 'flyrank-ai-internship-ml'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 118 (delta 32), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 1.86 MiB | 11.97 MiB/s, done.
Resolving deltas: 100% (32/32), done.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Task type:** ranking. An editor acts on a limited queue, so what matters is ordering candidates by "review this first," not a hard accept/reject label. Metric = precision@K.

**Chosen method:** Logistic Regression, escalated to Random Forest only if it earns it.

Why LR first:
- My task is a binary label with an observed outcome (recent-30d CTR underperformance), which maps to the skill's "start with Logistic Regression, then Random Forest."
- LR outputs a probability I can rank by, and its feature coefficients are readable — Section 4 needs me to name the top-3 drivers and check they make sense.

Why RF as the escalation: CTR is strongly non-linear vs position tier, and interactions matter (position x content type x volume). A forest can capture those. But "add complexity only when the comparison earns it" — so RF enters the table only if it beats LR on the same split and metric. If it doesn't, LR is the model and I say why more complexity bought nothing.

Methods intentionally not used, and why:
- Correlation/signal analysis — already done in w04; not a candidate ranker.
- K-Means clustering — Lane 3's tool; my lane has a defined target, so unsupervised grouping is off-task here.
- Single Decision Tree — too unstable on fuzzy CTR data; one shallow tree can't use all features.
- Gradient Boosting (XGB/LGBM) — "where safe" per the card; higher overfit risk on this small pool and worse interpretability. Not earned on round one; noted, not attempted.
- SVM / KNN / Naive Bayes — no clean probabilities for ranking; irrelevant to a ranking question here.

Model family: scikit-learn classifiers, ranked by predicted P(label), scored with precision@K and average precision against the same hold-out label.

In [10]:
# Method choice in one line: Logistic Regression first (readable, probabilistic),
# Random Forest as an escalation check -- the skill says add complexity only when
# the comparison earns it, so RF enters the table only if it beats LR on this split.
import pandas as pd

df = pd.read_csv(r"data/raw/content_refresh_anonymized.csv")
print("Planning input: one row = one content item.")
print(f"Rows: {len(df):,} | Clients: {df['client_id'].nunique()}")

# The candidates we will actually fit in Section 3:
MODELS = {
    "logistic_regression": True,   # start here
"random_forest": True,         # escalation, only report if it beats LR on same split
}
print("Confirmed model plan:", list(MODELS.keys()))

pd.Series({
    "task_type": "binary label -> ranked queue",
    "start": "logistic_regression",
    "escalate_to": "random_forest (only if it earns it)",
    "metric": "precision@K, average precision, base rate",
}).to_frame("choice")

Planning input: one row = one content item.
Rows: 30,000 | Clients: 32
Confirmed model plan: ['logistic_regression', 'random_forest']


,choice
task_type,binary label -> ranked queue
start,logistic_regression
escalate_to,random_forest (only if it earns it)
metric,"precision@K, average precision, base rate"


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped holdout by client_id — the honest split for this question.**

Pages from the same client share a site: the same template, content-type mix, keyword strategy, and tracking setup. If I shuffle rows, a client's pages can land on both sides of the split, and the model can memorize that site's quirks instead of learning from signals. The test score would be inflated — the model would look like it learned CTR when it actually remembered the client. Since my lane asks *"can signals flag underperformers on pages/sites we haven't tuned on?"*, whole clients must be held out.

**Skew fix:** clients are not equal-sized (largest 5,690 rows, smallest 1, median 148). Holding out by client count would either starve training (a few big clients carry most rows) or make the test set trivially tiny. So I hold out by row-share: pick clients whose rows sum to ~20% of the pool.

**Window guard (the leak check):** the label lives in the recent-30d window, so features come only from the prior-30d window plus static metadata and position. No `ctr`, no 90-day totals, no `*_last_30d` columns, no `trend_*`. The model can never read the recent-30d CTR that defines the label.

In [11]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Pool: pages with enough prior-window visibility, a known position, and a computable
# recent-30d CTR (the label window).
pool = df[
    (df["impressions_prev_30d"] >= 100)
    & (df["avg_position"] > 0)
    & (df["impressions_last_30d"] > 0)
].copy()

# Label: recent-30d CTR in the pooled bottom quartile.
pool["ctr_last30"] = pool["clicks_last_30d"] / pool["impressions_last_30d"] * 100
q = pool["ctr_last30"].quantile(0.25)
pool["label"] = (pool["ctr_last30"] <= q).astype(int)

print(f"Pool: {len(pool):,} rows | {pool['client_id'].nunique()} clients | base rate {pool['label'].mean():.1%}")

# Deterministic, size-weighted grouped holdout: hold out the client set whose rows sum
# closest to ~20% of the pool, straddling sizes (a couple of mid-size clients).
HOLD_OUT = ["client_4e07408562", "client_3fdba35f04"]
test = pool[pool["client_id"].isin(HOLD_OUT)]
train = pool[~pool["client_id"].isin(HOLD_OUT)]

print(f"Train: {len(train):,} rows / {train['client_id'].nunique()} clients / base rate {train['label'].mean():.1%}")
print(f"Test : {len(test):,} rows / {test['client_id'].nunique()} clients / base rate {test['label'].mean():.1%}")
print(f"Held-out row share: {len(test) / len(pool):.1%} (target ~20%)")
assert abs(train["label"].mean() - test["label"].mean()) < 0.10, "base rates too different -- check split"
print("Base-rate sanity check: passed (train vs test label share within 10pp).")

Pool: 17,956 rows | 30 clients | base rate 39.2%
Train: 14,104 rows / 28 clients / base rate 38.6%
Test : 3,852 rows / 2 clients / base rate 41.3%
Held-out row share: 21.5% (target ~20%)
Base-rate sanity check: passed (train vs test label share within 10pp).


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The rule baseline is re-computed **in this notebook, on this same split** — not reused from w04 — because the skill demands the baseline appear in the same table as the model, fit under the same train/test discipline. The label, the split, and every feature are identical across all three rows, so the comparison is apples-to-apples.

**Leak-safe feature contract:** the label is recent-30d CTR, so a feature may only come from the prior-30d window, static metadata, or position. `ctr`, all `*_last_30d`, all 90-day totals, and `trend_*` are excluded because they read the answer (recall `ctr = clicks / impressions` exposes the label directly).

In [12]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, average_precision_score

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
pool = df[
    (df['impressions_prev_30d'] >= 100)
    & (df['avg_position'] > 0)
    & (df['impressions_last_30d'] > 0)
].copy()
pool['ctr_last30'] = pool['clicks_last_30d'] / pool['impressions_last_30d'] * 100
q = pool['ctr_last30'].quantile(0.25)
pool['label'] = (pool['ctr_last30'] <= q).astype(int)

HOLD_OUT = ['client_4e07408562', 'client_3fdba35f04']
test = pool[pool['client_id'].isin(HOLD_OUT)].copy()
train = pool[~pool['client_id'].isin(HOLD_OUT)].copy()

# --- Shared, decision-point-safe feature definitions ------------------------
# The label is recent-30d CTR, so features may ONLY be prior-30d window + static
# metadata + position. Anything from the recent-30d or 90-day totals would read the
# answer (leakage): ctr = clicks/impressions exposes the label directly.
NUM_FEATS = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
             "avg_position", "content_age_days", "days_since_last_update",
             "search_volume", "competition", "cpc"]
CAT_FEATS = ["position_tier", "content_type", "main_intent",
             "competition_level", "word_count_tier"]

# word_count is missing for ~26% of rows and that missingness tracks content_type,
# so a blind fillna(0) would silently inject the content-type signal. Instead we
# carry an explicit flag so the model can use missingness as its own signal.
FLAG_FEATS = ["has_word_count"]
pool["has_word_count"] = pool["word_count"].notna().astype(int)
train["has_word_count"] = train["word_count"].notna().astype(int)
test["has_word_count"] = test["word_count"].notna().astype(int)

FEATURES = NUM_FEATS + CAT_FEATS + FLAG_FEATS

# One preprocessing pipeline. It is FIT ON TRAIN ONLY: fitting the imputer/scaler on
# the full pool would let training peek at test statistics -- the same leak we avoid
# for the model itself.
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), NUM_FEATS),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT_FEATS),
    ('flag', 'passthrough', FLAG_FEATS),
])

X_train = pd.DataFrame(train, columns=FEATURES)
y_train = train['label']
X_test = pd.DataFrame(test, columns=FEATURES)
y_test = test['label']

In [13]:
# --- Baseline (the w04 idea, re-versioned to the PRIOR window) --------------
# Tier-expected CTR is computed from the TRAIN split only, so the baseline can't
# peek at test rows (same discipline as the model). The score is the old rule but on
# prior-window inputs, so it no longer reads the recent-30d label.
tierexp = (train.groupby('position_tier', observed=True).apply(
    lambda g: g['clicks_prev_30d'].sum() / g['impressions_prev_30d'].sum() * 100,
    include_groups=False))
test_bl = test.copy()
test_bl['ctr_prev30'] = test_bl['clicks_prev_30d'] / test_bl['impressions_prev_30d'] * 100
test_bl['tier_exp'] = test_bl['position_tier'].map(tierexp)
test_bl['baseline_score'] = (test_bl['tier_exp'] - test_bl['ctr_prev30']) * np.log1p(test_bl['impressions_prev_30d'])
print('Baseline tier-expected CTR (train):')
print(tierexp.round(3).to_string())

Baseline tier-expected CTR (train):
position_tier
deep        0.052
page_1      0.344
page_3_5    0.150
striking    0.383
top_3       0.594


In [14]:
# --- One honest scorer: rank by score/probability, measure precision@K + AP -------
# This is 'how many of my top-K review recommendations were actually underperformers'.
# Test has 3,852 rows at a 41% base rate; @20/@50/@100 match a team's review capacity.
def precision_at_k(y_true, score, k):
    idx = np.argsort(score)[::-1][:k]      # rank descending
    return precision_score(y_true.iloc[idx].values, np.ones(k, dtype=int))  # top-k share positive

def summarize(y_true, score, name):
    cols = {'P@20': precision_at_k(y_true, score, 20),
            'P@50': precision_at_k(y_true, score, 50),
            'P@100': precision_at_k(y_true, score, 100),
            'AP': average_precision_score(y_true, score)}
    return pd.Series(cols, name=name)

# --- 1) Baseline -------------------------------------------------------------
res_baseline = summarize(y_test, test_bl['baseline_score'], 'Baseline (prior-tier CTR gap)')

# --- 2) Logistic Regression first (readable, this is 'the model' until RF earns it)
lr = make_pipeline(pre, LogisticRegression(max_iter=3000, random_state=42))
lr.fit(X_train, y_train)
pred_lr = lr.predict_proba(X_test)[:, 1]
res_lr = summarize(y_test, pred_lr, 'Logistic Regression')

# --- 3) Random Forest ONLY as an escalation check -----------------------------
# 'Add complexity only when the comparison earns it.' Fit it, add its row to the
# same table; it keeps its row even if it loses, but we name the winner honestly.
rf = make_pipeline(pre, RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
rf.fit(X_train, y_train)
pred_rf = rf.predict_proba(X_test)[:, 1]
res_rf = summarize(y_test, pred_rf, 'Random Forest')

table = pd.concat([res_baseline, res_lr, res_rf], axis=1).T.round(3)
table['Base rate'] = round(y_test.mean(), 3)
print(table)
print(f"\nTest rows: {len(y_test):,} | Test base rate (share of true underperformers): {y_test.mean():.1%}")

                               P@20  P@50  P@100     AP  Base rate
Baseline (prior-tier CTR gap)  0.30  0.36   0.37  0.447      0.413
Logistic Regression            0.80  0.86   0.87  0.777      0.413
Random Forest                  0.85  0.88   0.92  0.767      0.413

Test rows: 3,852 | Test base rate (share of true underperformers): 41.3%


In [15]:
# --- Verdict: did learning beat the rule? ---------------------------------------
# Compare each model's AP and P@K to the baseline on the SAME test rows. We only
# crown the model that actually wins; if RF ties/loses to LR, we keep its row but
# say LR is the model and more complexity bought nothing.
best_model = 'Logistic Regression' if res_lr['AP'] >= res_rf['AP'] else 'Random Forest'
print('LR beats baseline at AP?', bool(res_lr['AP'] >= res_baseline['AP']))
print('RF beats LR at AP?', bool(res_rf['AP'] >= res_lr['AP']))
print('Highest AP:', best_model)

LR beats baseline at AP? True
RF beats LR at AP? False
Highest AP: Logistic Regression


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

A metric without error analysis is decoration. Three questions, answered honestly:

1. **What does the model lean on?** Permutation importance on the held-out test set — the drop in AP when each feature is randomly shuffled. Then a sanity check: does the top feature make sense, or is it suspiciously perfect (= leakage)?
2. **Where is it most wrong?** Split the held-out errors and look for patterns — which tiers and value ranges trip it up.
3. **What do the wrong cases look like?** Show a few concrete misclassified pages and say why they are genuinely hard.

In [17]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score

# --- 1) Permutation importance (drop in test AP when each feature is shuffled) ---
# Unlike raw coefficients, this measures how much the RANKING depends on each feature on
# data the model has never seen. Computed on the TEST set -- the same rows as the metrics
# table -- so it describes the deployed model, not a cross-validation artifact.
X_test_enc = pre.transform(X_test)
feat_names = np.array(NUM_FEATS + ['cat:' + c for c in CAT_FEATS] + FLAG_FEATS)
perm = permutation_importance(lr, X_test, y_test, scoring='average_precision',
                              n_repeats=10, random_state=42, n_jobs=-1)

imp = pd.DataFrame({'feature': feat_names, 'mean_AP_drop': perm.importances_mean})\
    .sort_values('mean_AP_drop', ascending=False)
print('Top-8 features by permutation importance (test AP drop when shuffled):')
print(imp.head(8).round(4).to_string(index=False))

print('\nSanity check: the top driver is a prior-window signal or static metadata -- it predicts')
print('future underperformance without reading the recent-30d CTR (the label). No ctr / *_last_30d')
print('/ trend_* in the list, so nothing suspiciously perfect (no leakage).')

Top-8 features by permutation importance (test AP drop when shuffled):
             feature  mean_AP_drop
     clicks_prev_30d        0.1272
impressions_prev_30d        0.1032
        avg_position        0.0772
    content_age_days        0.0067
         competition        0.0059
      has_word_count        0.0037
   sessions_prev_30d        0.0035
     cat:main_intent        0.0030

Sanity check: the top driver is a prior-window signal or static metadata -- it predicts
future underperformance without reading the recent-30d CTR (the label). No ctr / *_last_30d
/ trend_* in the list, so nothing suspiciously perfect (no leakage).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.